# Hourly HSS catalogs — B1

**Method source:** `/Users/aosh/Developer/sdds/downloader/dbscripts/hss/modelB1.sql`; gap filling: `/Users/aosh/Developer/sdds/downloader/dbscripts/hss/ace_fill.sql`.

For the merged hourly parquets (`V` in km/s, `N_p` in cm⁻³, `T_p` in K), flag `V > 400`, `N_p < 6.1`, and `T_p > 82000`. Keep these flags and add hours with >12 raw positives in a centered 25-hour window. Retain events spanning ≥24 h, or ≥12 h with peak speed >450 km/s.

The SQL gap fill is preserved: interpolate missing values in groups of 2–25 rows with valid endpoints; otherwise substitute zero. Windows are evaluated chronologically. Isolated flags and the first/last grid samples are excluded as in SQL. Thresholds are applied unchanged to all satellites.

Outputs: `SAT_NAME_hss_b1.csv` with `T_start`, `T_end`, `T_max` (earliest peak; UTC), and `v_max` (km/s).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
archive_dir = Path("/Users/aosh/Developer/Shock-and-Awe/Data/CDAWeb Archive/merged")
parquet_paths = sorted(archive_dir.glob("*.parquet"))
output_dir = Path("/Users/aosh/Developer/helio-n/Outputs/HSS B1")

In [3]:
def find_hss_b1(plasma):
    """Apply the chronological SDDS B1 algorithm to hourly plasma columns."""
    plasma = plasma.loc[:, ["speed", "density", "temperature"]].copy()
    plasma.index = pd.to_datetime(plasma.index, utc=True)
    plasma = plasma.sort_index().astype(float)
    plasma = plasma.reindex(pd.date_range(plasma.index.min(), plasma.index.max(), freq="h"))

    # Port ace_fill.sql's group definition and endpoint tests, including zero fill.
    filled = plasma.copy()
    positions = pd.Series(np.arange(len(plasma)), index=plasma.index)
    for column in plasma.columns:
        values = plasma[column].to_numpy()
        present = plasma[column].notna()
        groups = (present & present.shift(1, fill_value=False)).cumsum()
        grouped_positions = positions.groupby(groups)
        first = grouped_positions.transform("min").to_numpy()
        last = grouped_positions.transform("max").to_numpy()
        count = grouped_positions.transform("size").to_numpy()
        missing = np.isnan(values)
        interpolate = missing & (count > 1) & (count < 26) & np.isfinite(values[first]) & np.isfinite(values[last])
        result = values.copy()
        result[missing] = 0.0
        result[interpolate] = values[first[interpolate]] + (
            values[last[interpolate]] - values[first[interpolate]]
        ) * (positions.to_numpy()[interpolate] - first[interpolate]) / (last[interpolate] - first[interpolate])
        filled[column] = result

    raw = (filled.speed > 400) & (filled.density < 6.1) & (filled.temperature > 82000)
    marked = raw | (raw.rolling(25, center=True, min_periods=1).sum() > 12)
    groups = (~marked).cumsum()

    # Reproduce SQL's s/b/e labels: adjacent positives, with both neighbors defined.
    retained = marked & (marked.shift(1, fill_value=False) | marked.shift(-1, fill_value=False))
    retained.iloc[0] = False
    retained.iloc[-1] = False
    selected = filled.loc[retained, ["speed"]].copy()
    selected["dt"] = selected.index
    selected["group"] = groups.loc[retained]
    events = selected.groupby("group", sort=False).agg(
        start_dt=("dt", "min"),
        end_dt=("dt", "max"),
        vmax_dt=("speed", "idxmax"),
        vmax=("speed", "max"),
        samples=("speed", "size"),
    )
    duration_h = (events.end_dt - events.start_dt).dt.total_seconds() / 3600
    keep = (events.samples >= 12) & (
        (duration_h >= 24) | ((duration_h >= 12) & (events.vmax > 450))
    )
    return events.loc[keep, ["start_dt", "end_dt", "vmax_dt", "vmax"]].rename(
        columns={"start_dt": "T_start", "end_dt": "T_end", "vmax_dt": "T_max", "vmax": "v_max"}
    ).reset_index(drop=True)

In [4]:
output_dir.mkdir(parents=True, exist_ok=True)
catalogs = {}
for parquet_path in parquet_paths:
    satellite = parquet_path.stem.split("_")[0]
    plasma = pd.read_parquet(parquet_path, columns=["V", "N_p", "T_p"]).rename(
        columns={"V": "speed", "N_p": "density", "T_p": "temperature"}
    )
    catalogs[satellite] = find_hss_b1(plasma)
    catalogs[satellite].to_csv(output_dir / f"{satellite}_hss_b1.csv", index=False)

pd.Series({satellite: len(events) for satellite, events in catalogs.items()}, name="events")

ace         672
psp           0
solo        105
stereo-a    603
stereo-b    153
Name: events, dtype: int64

In [5]:
hss_events = pd.concat(catalogs, names=["satellite", "event"])
hss_events

T_start                     T_end  \
satellite event                                                       
ace       0     2010-01-11 16:00:00+00:00 2010-01-12 04:00:00+00:00   
          1     2010-01-13 07:00:00+00:00 2010-01-14 17:00:00+00:00   
          2     2010-01-20 23:00:00+00:00 2010-01-21 22:00:00+00:00   
          3     2010-02-01 10:00:00+00:00 2010-02-03 17:00:00+00:00   
          4     2010-02-18 12:00:00+00:00 2010-02-19 14:00:00+00:00   
...                                   ...                       ...   
stereo-b  148   2014-08-27 21:00:00+00:00 2014-08-30 01:00:00+00:00   
          149   2014-09-02 16:00:00+00:00 2014-09-05 09:00:00+00:00   
          150   2014-09-07 11:00:00+00:00 2014-09-09 09:00:00+00:00   
          151   2014-09-14 00:00:00+00:00 2014-09-15 06:00:00+00:00   
          152   2014-09-16 13:00:00+00:00 2014-09-17 17:00:00+00:00   

                                    T_max       v_max  
satellite event                                        
ace       0     2010-01-11 18:00:00+00:00  492.489990  
          1     2010-01-14 01:00:00+00:00  513.219971  
          2     2010-01-21 00:00:00+00:00  510.459991  
          3     2010-02-03 02:00:00+00:00  557.679993  
          4     2010-02-19 08:00:00+00:00  487.769989  
...                                   ...         ...  
stereo-b  148   2014-08-29 05:00:00+00:00  446.000000  
          149   2014-09-03 10:00:00+00:00  772.599976  
          150   2014-09-08 14:00:00+00:00  541.900024  
          151   2014-09-14 02:00:00+00:00  526.299988  
          152   2014-09-17 05:00:00+00:00  431.200012  

[1533 rows x 4 columns]